### Hugging Face
- 이미지캡셔닝   :  Salesforce/blip-image-captioning-base
- 텍스트 임베딩  :  sentence-transformers/all-MiniLM-L6-v2
- 텍스트 생성    : distilbert/distilgpt2  (경량 모델)

In [1]:
import os
import fitz
import chromadb
from PIL import Image
from transformers import pipeline,BlipProcessor, BlipForConditionalGeneration
from chromadb.utils import embedding_functions

In [2]:
# BLIP 모델 로드
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [3]:
# 임베딩 모델 로드
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='sentence-transformers/all-MiniLM-L6-v2')

c:\Users\Playdata\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# vectorDB 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db_hf')
collection = chroma_client.get_or_create_collection(name='multimocal_rag_hf',embedding_function=hf_ef)

### 데이터추출 및 오픈소스 임베딩
- PDF 텍스트와 추출된 이미지를 BLIP 모델로 갭셔닝한 뒤, Sentence Transformers임베딩을 사용해서 로컬 vectorDB에 저장

In [5]:
ptf_path = 'sample_paper.pdf'
if collection.count() == 0:
    doc = fitz.open(ptf_path)
    documents,metadatas,ids = [],[],[]
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추가
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metadatas.append({'type':'text','page':page_num+1})
            ids.append(f'hf_text_page_{page_num+1}')
        # 이미지 캡셔닝
        image_lists = page.get_images(full=True)
        if image_lists:
            xref = image_lists[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            image_filename = f'ht_temp_img.{ext}'
            with open(image_filename, 'wb') as f:
                f.write(image_bytes)
            
            try:
                raw_image = Image.open(image_filename).convert('RGB')
                inputs = processor(raw_image, return_tensors='pt')
                out = model.generate(**inputs,max_new_tokens=50)
                caption = processor.decode(out[0],skip_special_tokens=True)

                documents.append(caption)
                metadatas.append({'type':'image_summary', 'page':page_num+1})
                ids.append(f'hf_image_summary_page_{page_num+1}')
                print(f'페이지 {page_num+1} 이미지 캡셔닝 완료 : {caption}')
            except Exception as e:
                print(f'페이지 {page_num+1} 이미지 캡셔닝 실패 : {e}')
    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)
else:
    print(collection.count())


페이지 3 이미지 캡셔닝 완료 : a block diagram with the block and block
페이지 4 이미지 캡셔닝 완료 : a diagram of the same algorithm


### RAG 검색 및 LLM을 통한 답변 생성

In [6]:
query = 'what is transformer architecture?'
results = collection.query(
    query_texts=[query],
    n_results=1
)
retrieved_docs = results['documents'][0]
retrieved_metas = results['metadatas'][0]
print(f'검색된 문서 수 : {len(retrieved_docs)}')
for meta in retrieved_metas:
    print(f" - [출처:페이지{meta['page']}, 타입] : {meta['type']}")
context = '\n\n--\n\n'.join(retrieved_docs) 
print('\n오픈소스 답변 생성중\n')
generator = pipeline('text-generation', model='distilbert/distilgpt2')
prompt = f'Context:{context[:400]}\n\nQuestion:{query}\n\nAnswer:'
response = generator(prompt,max_new_tokens=100, num_return_sequences=1)
print('최종답변')
print(response[0]['generated_text'].replace(prompt, "").strip())

검색된 문서 수 : 1
 - [출처:페이지3, 타입] : text

오픈소스 답변 생성중



config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--distilbert--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


최종답변
The transformer is a stack of N = 6 identical layers. The
Sizer is an unconnected
sub-layers of N = 6 identical layers. The
sizer is an unconnected
sub-layers of N = 6 identical layers. The
sizer is an unconnected
sub-layers of N = 6 identical layers. The
Sizer is an unconnected
sub-layers of N = 6 identical layers.
The sub-layers of


- RAG(이미지 인식->임베딩->검색->답변생성) 가능
- 모델 성능차이.. 경량화 모델인 BLIP는 문맥이해력이 낮아서 반복적인 텍스트를 출력
- 향후개선 : VRAM확보, 이미지인식모델은 LLaVA-1.5 이상,, 생성모델 Llama-3(8B) 수준으로 교체

In [7]:
# https://ollama.com/download/windows